# Feature Extractor Pipeline Test

Runs `src/feature_extractor.py` end-to-end on a small sample so you can validate the full extraction + temporal feature pipeline.

Edit the config cell, then run all cells in order.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import shlex
import subprocess
import sys

import pandas as pd

try:
    import torch
    HAS_CUDA = bool(torch.cuda.is_available())
except Exception:
    HAS_CUDA = False

CANDIDATE_ROOTS = [
    Path('/playpen-ssd/smerrill/deception2'),
    Path('/work/users/s/m/smerrill/deception2'),
]
ROOT = next((p for p in CANDIDATE_ROOTS if p.exists()), CANDIDATE_ROOTS[0])
SCRIPT_PATH = ROOT / 'src' / 'feature_extractor.py'
NOTEBOOK_TMP = ROOT / 'Notebooks' / '_tmp'
NOTEBOOK_TMP.mkdir(parents=True, exist_ok=True)

print(f'ROOT: {ROOT}')
print(f'SCRIPT_PATH: {SCRIPT_PATH}')
print(f'HAS_CUDA: {HAS_CUDA}')


In [ ]:
# Dataset/model candidates. The first complete one is used.
DATASET_CANDIDATES = [
    {
        'name': 'BS-7B',
        'examples': ROOT / 'Dataset' / 'BS' / 'DeepSeek-R1-Distill-Qwen-7B' / 'examples.jsonl',
        'sentences': ROOT / 'Dataset' / 'BS' / 'DeepSeek-R1-Distill-Qwen-7B' / 'sentences.jsonl',
        'localization': ROOT / 'Dataset' / 'BS' / 'DeepSeek-R1-Distill-Qwen-7B' / 'localization.jsonl',
        'model_name': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B',
    },
    {
        'name': 'Gridworld-7B',
        'examples': ROOT / 'Dataset' / 'Gridworld' / 'DeepSeek-R1-Distill-Qwen-7B' / 'examples.jsonl',
        'sentences': ROOT / 'Dataset' / 'Gridworld' / 'DeepSeek-R1-Distill-Qwen-7B' / 'sentences.jsonl',
        'localization': ROOT / 'Dataset' / 'Gridworld' / 'DeepSeek-R1-Distill-Qwen-7B' / 'localization.jsonl',
        'model_name': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B',
    },
]

DATASET = None
for candidate in DATASET_CANDIDATES:
    if candidate['examples'].exists() and candidate['sentences'].exists() and candidate['localization'].exists():
        DATASET = candidate
        break

if DATASET is None:
    raise FileNotFoundError('No valid dataset candidate found. Update DATASET_CANDIDATES.')

RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = NOTEBOOK_TMP / f'feature_extractor_test_{RUN_TAG}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Main knobs for a smoke test.
NUM_EXAMPLES = 16
SEED = 0
DEVICE = 'cuda' if HAS_CUDA else 'cpu'
MAX_TOKENS = 4096
TOPK_VOCAB = 32
PROGRESS_EVERY = 5

OUT_PATH = RUN_DIR / 'temporal_features.parquet'
RAW_OUT_PATH = RUN_DIR / 'raw_features.parquet'
FEATURE_SETS_OUT = RUN_DIR / 'feature_sets.json'
MANIFEST_OUT = RUN_DIR / 'manifest.json'

print(f"Dataset: {DATASET['name']}")
print(f"examples: {DATASET['examples']}")
print(f"sentences: {DATASET['sentences']}")
print(f"localization: {DATASET['localization']}")
print(f"model_name: {DATASET['model_name']}")
print(f"device: {DEVICE}")
print(f"run_dir: {RUN_DIR}")


In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    '--out-path', str(OUT_PATH),
    '--examples-path', str(DATASET['examples']),
    '--sentences-path', str(DATASET['sentences']),
    '--localization-path', str(DATASET['localization']),
    '--model-name', DATASET['model_name'],
    '--num-examples', str(NUM_EXAMPLES),
    '--seed', str(SEED),
    '--only-localized',
    '--max-tokens', str(MAX_TOKENS),
    '--topk-vocab', str(TOPK_VOCAB),
    '--device', DEVICE,
    '--progress-every', str(PROGRESS_EVERY),
    '--raw-out-path', str(RAW_OUT_PATH),
    '--feature-sets-out', str(FEATURE_SETS_OUT),
    '--manifest-out', str(MANIFEST_OUT),
]

print('Command:')
print(' '.join(shlex.quote(x) for x in cmd))


In [ ]:
# Set to False for a dry run (print command only).
RUN_PIPELINE = True

if RUN_PIPELINE:
    proc = subprocess.run(cmd, check=False)
    if proc.returncode != 0:
        raise RuntimeError(f'Pipeline failed with return code {proc.returncode}')
else:
    print('Skipping execution because RUN_PIPELINE=False')


In [ ]:
assert OUT_PATH.exists(), f'Missing output parquet: {OUT_PATH}'
assert FEATURE_SETS_OUT.exists(), f'Missing feature sets json: {FEATURE_SETS_OUT}'
assert MANIFEST_OUT.exists(), f'Missing manifest json: {MANIFEST_OUT}'

df = pd.read_parquet(OUT_PATH)
with FEATURE_SETS_OUT.open('r', encoding='utf-8') as f:
    feature_sets = json.load(f)
with MANIFEST_OUT.open('r', encoding='utf-8') as f:
    manifest = json.load(f)

print(f'Output rows/cols: {df.shape}')
print('Feature set sizes:')
for k, v in feature_sets.items():
    print(f'  {k}: {len(v)}')

required_cols = ['example_id', 'sentence_idx', 'deception_rate']
missing = [c for c in required_cols if c not in df.columns]
print(f'Missing required cols: {missing}')

display(df.head(10))
display(pd.Series(manifest).head(20))


## Next Tweaks

- Increase `NUM_EXAMPLES` for a fuller run.
- Change `MODEL_NAME`/dataset paths if you want to test another setup.
- Tune temporal args by appending flags to `cmd` in the command cell.